In [1]:
import vectorbt as vbt
import pandas as pd
import numpy as np
from datetime import datetime

### **CONFIG**

In [ ]:
# ================== CONFIG ==================
SYMBOL = "QTUM"          # or "QTUM" ETF
VIX_SYMBOL = "^VIX"
START_DATE = "2018-01-01"   # QTUM inception ~2018
END_DATE = datetime.now().strftime("%Y-%m-%d")

# MA periods
FAST_MA = 50
SLOW_MA = 200

# VIX thresholds (test multiple in sweep)
VIX_ENTER = 20
VIX_EXIT = 20   # or 25 for stricter regime filter

# Risk controls (your style)
INITIAL_CASH = 100_000
FEES = 0.001      # 0.1% round-trip approx for ETF
SLIPPAGE = 0.001

### **DATA LOADING**

In [ ]:
# ================== DATA LOADING ==================

print("Downloading data...")

data = vbt.YFData.download(
    [SYMBOL, VIX_SYMBOL],
    start=START_DATE,
    end=END_DATE,
    interval="1d"
).get("Close")   # or use OHLCV if needed

price = data[SYMBOL]
vix = data[VIX_SYMBOL]

# Align dates
price, vix = price.align(vix, join='inner')

### **INDICATORS**

In [ ]:
# ================== INDICATORS ==================
# Moving Averages
ma_fast = vbt.MA.run(price, window=FAST_MA, ewm=False)
ma_slow = vbt.MA.run(price, window=SLOW_MA, ewm=False)

# Golden / Death Cross signals
golden_cross = ma_fast.ma_crossed_above(ma_slow)
death_cross = ma_fast.ma_crossed_below(ma_slow)

# VIX Regime
in_calm_regime = vix < VIX_ENTER
in_risk_off = vix >= VIX_EXIT

### **ENTRY / EXIT SIGNALS**

In [ ]:
# ================== ENTRY / EXIT SIGNALS ==================
# Long when: Golden Cross AND calm regime
# (You can also use sustained condition: ma_fast.ma > ma_slow AND calm)
entries = golden_cross & in_calm_regime

# Exit when: Death Cross OR risk-off
exits = death_cross | in_risk_off

# Optional: Stay in position only while trend + regime both favorable
# trend_up = ma_fast.ma > ma_slow
# entries = trend_up & in_calm_regime
# exits = ~trend_up | in_risk_off

### **BACKTEST**

In [ ]:
# ================== BACKTEST ==================
pf = vbt.Portfolio.from_signals(
    price,
    entries=entries,
    exits=exits,
    init_cash=INITIAL_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    direction="longonly",
    freq="1D"
)

### **RESULTS**

In [ ]:
# ================== RESULTS ==================
print(pf.stats())
pf.plot().show()                    # Equity curve, drawdowns, trades
# pf.orders.plot()                  # Trade visualization
# pf.drawdown.plot()                # Drawdown analysis